# logit-thermo A2-4：分析（CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 5〜15 分。
- `MyDrive/logit-thermo/A2_4/` の 9 回分と、A2-2 の 3 モデル（確かめ用）を読み、ゲート v0.2 の判定を計算して `A2_4/a2_4_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：`scripts/a2_2_analyze.py`（`35142703f6b88d7a…`）と `scripts/a2_4_analyze.py`（`e70bad0db9b7780c…`）をそのまま埋め込み（データを開く前に固定）。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo A2-4 分析（判定と要約、CPU）」。終わったらランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-4 分析（判定と要約、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-4 analyze', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：分析のコード
import hashlib
SRCA22 = '"""A2-2（ゲート v0.3、凍結 2026-10-27 21:34）の分析。データを開く前に書いて固定する。\n\n    python3 scripts/a2_2_analyze.py <A2_2 のフォルダ> <出力 json>\n\nゲートに書いていない細部（データを開く前にここで決めた。結果ノートにも写す）：\n- F(T)：書き出しごとに、生成 2 本の ℓ_gen を平均してから ℓ_tf を引き、t = 2〜512 で平均。書き出しどうしの中央値。\n- ε：格子の各温度で、書き出し単位のブートストラップ（2,000 回、種 0）による F の 95% 区間の半分の幅。その中央値。\n  T_0 のブートストラップでは ε を全データの値に固定する（入れ子のブートストラップはしない）。\n  交差しない回は区間の計算から外し、その割合を記録する。\n- T_cal：全書き出しの Σ(⟨z⟩_T − z_y) を足し、符号が + から − に変わるところを対数温度で直線補間。\n  有限でない値を含む書き出し（人のトークンが外した番号だった場合）は T_cal から外して記録する。\n- a：a_loc の、全位置・全書き出しの中央値。\n- 対照 B は「ほかの家族の T_0 の平均（算術）」、C は「a × ほかの家族の T_0 ÷ a の平均」。どちらも交差のあるモデルだけで作る。\n- E2 の「誤差 10% 以内」は |log(T_0 ÷ T_cal)| ≤ log(1.10)。「区間の幅 15% 以下」は 区間の上 ÷ 下 ≤ 1.15。\n- E1：どちらかの組で T_0 がない（交差なし）ときは「交差なしで判定できない」と書く。\n- P2：横軸 x1 = log T、x3 = ℓ_tf(T)（全位置・全書き出しの中央値）。全 7 モデルの 15 点を使う。\n  3 次の最小二乗スプライン（scipy LSQUnivariateSpline、k = 3）。内側の結び目は 5 つで、学習に使う x の分位点\n  （1/6, …, 5/6）に置く。x1 と x3 で柔らかさをそろえるため。同じ x は平均してまとめる。\n  学習の範囲の外は端の値で止める（ext = \'const\'）。範囲の外の点の割合を記録する。\n  （最初は平滑化スプライン s = 点の数 × σ² にしていたが、作り物のデータの確かめで、x3 の近い点が並ぶと\n   スプラインが暴れて x3 だけが不公平に負けることが分かったので、データを開く前に替えた。）\n  誤差は「抜いた家族の各モデルの二乗誤差の平均」の、全モデルでの平均。\n- Q6（記録）：自分の文への特別な反応 A = |ℓ_gen − ℓ_cross|、変な文だから B = |ℓ_cross − ℓ_tf(文字の窓)|。\n  ℓ_gen は読み手自身の生成の t = 2〜512 の平均、ℓ_cross と ℓ_tf(文字の窓) は対照のファイルの値。\n  4 つの向き × 15 温度で平均した A と B を比べ、|A − B| ≤ 0.2 × max(A, B) なら「同じくらい」。\n"""\nimport json, math, os, sys\nimport numpy as np\nfrom scipy.interpolate import LSQUnivariateSpline\n\nMODELS = [(\'pythia-1b\', \'Pythia\', \'base\'), (\'qwen2.5-0.5b\', \'Qwen\', \'base\'), (\'qwen2.5-0.5b-inst\', \'Qwen\', \'inst\'),\n          (\'qwen2.5-1.5b\', \'Qwen\', \'base\'), (\'olmo2-1b\', \'OLMo\', \'base\'), (\'smollm2-1.7b\', \'SmolLM\', \'base\'),\n          (\'smollm2-1.7b-inst\', \'SmolLM\', \'inst\')]\nPAIRS = [(\'qwen2.5-0.5b\', \'qwen2.5-0.5b-inst\'), (\'smollm2-1.7b\', \'smollm2-1.7b-inst\')]\nCROSS = [(\'pythia-1b\', \'olmo2-1b\'), (\'olmo2-1b\', \'pythia-1b\'), (\'qwen2.5-1.5b\', \'smollm2-1.7b\'), (\'smollm2-1.7b\', \'qwen2.5-1.5b\')]\nNBOOT, SEED = 2000, 0\nLOG110 = math.log(1.10)\n\n\ndef pav(y):\n    blocks = [[float(v), 1] for v in y]\n    i = 0\n    while i < len(blocks) - 1:\n        if blocks[i][0] > blocks[i + 1][0]:\n            m = (blocks[i][0] * blocks[i][1] + blocks[i + 1][0] * blocks[i + 1][1]) / (blocks[i][1] + blocks[i + 1][1])\n            blocks[i] = [m, blocks[i][1] + blocks[i + 1][1]]; del blocks[i + 1]\n            i = max(i - 1, 0)\n        else:\n            i += 1\n    out = []\n    for m, n in blocks:\n        out += [m] * n\n    return np.array(out)\n\n\ndef crossing_mono(F, temps, eps):\n    g = pav(np.asarray(F, float))\n    neg = np.nonzero(g <= -eps)[0]; pos = np.nonzero(g >= eps)[0]\n    if len(neg) == 0 or len(pos) == 0 or pos[0] < neg[-1]:\n        return float(\'nan\')\n    i, j = neg[-1], pos[0]\n    lt0, lt1 = math.log(temps[i]), math.log(temps[j])\n    return float(math.exp(lt0 + (0 - g[i]) / (g[j] - g[i]) * (lt1 - lt0)))\n\n\ndef per_doc_feedback(tf_q, gen_q):\n    """tf_q：(nD, nT, L)、gen_q：(nT, S, nD, L) → (nD, nT)：t = 2〜L の f の平均。"""\n    g = gen_q.astype(np.float64).mean(1).transpose(1, 0, 2)        # (nD, nT, L)\n    f = g - tf_q.astype(np.float64)\n    return f[:, :, 1:].mean(-1)\n\n\ndef feedback_stats(fd, temps, rng):\n    nD = fd.shape[0]\n    F = np.median(fd, 0)\n    idx = rng.integers(0, nD, size=(NBOOT, nD))\n    Fb = np.median(fd[idx], 1)                                       # (NBOOT, nT)\n    half = (np.percentile(Fb, 97.5, 0) - np.percentile(Fb, 2.5, 0)) / 2\n    eps = float(np.median(half))\n    T0 = crossing_mono(F, temps, eps)\n    T0b = np.array([crossing_mono(Fb[b], temps, eps) for b in range(NBOOT)])\n    ok = np.isfinite(T0b)\n    ci = [float(np.percentile(T0b[ok], 2.5)), float(np.percentile(T0b[ok], 97.5))] if ok.sum() >= 20 else [float(\'nan\')] * 2\n    return dict(F=F.tolist(), F_ci_half=half.tolist(), eps=eps, F_pav=pav(F).tolist(), T0=T0, T0_ci=ci,\n                boot_cross_frac=float(ok.mean()))\n\n\ndef t_cal(tcal, tcal_T):\n    fin = np.isfinite(tcal).all(1)\n    S = tcal[fin].sum(0)\n    lt = np.log(tcal_T)\n    i = np.nonzero((S[:-1] > 0) & (S[1:] <= 0))[0]\n    if len(i) == 0:\n        return float(\'nan\'), int((~fin).sum())\n    i = i[0]\n    return float(math.exp(lt[i] + S[i] / (S[i] - S[i + 1]) * (lt[i + 1] - lt[i]))), int((~fin).sum())\n\n\ndef distinct4(toks):\n    """toks：(nT, S, nD, L) → (nT,) 後半 128 トークンの distinct-4 の平均。"""\n    nT, S, nD, L = toks.shape\n    out = np.zeros(nT)\n    for k in range(nT):\n        v = []\n        for s in range(S):\n            for d in range(nD):\n                x = toks[k, s, d, L - 128:]\n                grams = {tuple(x[i:i + 4]) for i in range(len(x) - 3)}\n                v.append(len(grams) / (len(x) - 3))\n        out[k] = np.mean(v)\n    return out.tolist()\n\n\ndef spline_error(xs, Fs, fams, sigma):\n    errs, outside = [], []\n    for f in sorted(set(fams)):\n        tr = [i for i, g in enumerate(fams) if g != f]\n        te = [i for i, g in enumerate(fams) if g == f]\n        x = np.concatenate([xs[i] for i in tr]); y = np.concatenate([Fs[i] for i in tr])\n        ux = np.unique(x)\n        uy = np.array([y[x == v].mean() for v in ux])\n        knots = np.quantile(ux, np.arange(1, 6) / 6)\n        sp = LSQUnivariateSpline(ux, uy, knots, k=3, ext=\'const\')\n        for i in te:\n            errs.append(float(np.mean((sp(xs[i]) - Fs[i]) ** 2)))\n            outside.append(float(np.mean((xs[i] < ux.min()) | (xs[i] > ux.max()))))\n    return float(np.mean(errs)), float(np.mean(outside))\n\n\ndef judge(models, temps, q_tag):\n    """models：name → dict(fam, kind, fb, Tcal, a, ell_tf)。E1・E2・P2 と判定不能の条件。"""\n    out = {}\n    have = {n: m for n, m in models.items() if np.isfinite(m[\'fb\'][\'T0\'])}\n    T0s = {n: m[\'fb\'][\'T0\'] for n, m in have.items()}\n    # 判定不能の条件\n    if len(T0s) >= 2:\n        rng_ratio = max(T0s.values()) / min(T0s.values())\n        ci_ratios = [m[\'fb\'][\'T0_ci\'][1] / m[\'fb\'][\'T0_ci\'][0] for m in have.values() if np.isfinite(m[\'fb\'][\'T0_ci\'][0])]\n        typ = float(np.median(ci_ratios)) if ci_ratios else float(\'nan\')\n        undec_range = bool(np.isfinite(typ) and rng_ratio < typ)\n    else:\n        rng_ratio, typ, undec_range = float(\'nan\'), float(\'nan\'), True\n    tc = [m[\'Tcal\'] for m in models.values() if np.isfinite(m[\'Tcal\'])]\n    undec_D = bool(max(tc) / min(tc) < 1.10)\n    out[\'undecidable\'] = dict(T0_range_ratio=rng_ratio, T0_ci_ratio_typical=typ, E2_P2_undecidable=undec_range,\n                              Tcal_range_ratio=max(tc) / min(tc), E_vs_D_undecidable=undec_D)\n    # E1\n    e1 = []\n    for b, i in PAIRS:\n        dcal = math.log(models[i][\'Tcal\'] / models[b][\'Tcal\'])\n        if b in have and i in have:\n            d0 = math.log(T0s[i] / T0s[b])\n            ok = bool(np.sign(d0) == np.sign(dcal) and abs(d0 - dcal) <= 0.5 * abs(dcal))\n        else:\n            d0, ok = float(\'nan\'), None\n        e1.append(dict(pair=[b, i], d_cal=dcal, d_0=d0, ok=ok))\n    if any(p[\'ok\'] is None for p in e1):\n        v = \'交差なしで判定できない\'\n    elif all(abs(p[\'d_cal\']) < LOG110 for p in e1):\n        v = \'判定不能\'\n    else:\n        v = \'当たる\' if all(p[\'ok\'] for p in e1) else \'外れる\'\n    out[\'E1\'] = dict(pairs=e1, verdict=v)\n    # E2\n    rows = {}\n    for n, m in have.items():\n        others = [k for k in have if models[k][\'fam\'] != m[\'fam\']]\n        B = float(np.mean([T0s[k] for k in others])) if others else float(\'nan\')\n        C = m[\'a\'] * float(np.mean([T0s[k] / models[k][\'a\'] for k in others])) if others else float(\'nan\')\n        T0 = T0s[n]; lo, hi = m[\'fb\'][\'T0_ci\']\n        narrow = bool(np.isfinite(lo) and hi / lo <= 1.15)\n        rows[n] = dict(T0=T0, Tcal=m[\'Tcal\'], err_E=abs(math.log(T0 / m[\'Tcal\'])), err_B=abs(math.log(T0 / B)),\n                       err_C=abs(math.log(T0 / C)), err_D=abs(math.log(T0)), ci=[lo, hi], narrow_ci=narrow,\n                       within=bool(abs(math.log(T0 / m[\'Tcal\'])) <= LOG110 or (narrow and lo <= m[\'Tcal\'] <= hi)))\n    if rows:\n        mean = {k: float(np.mean([r[\'err_\' + k] for r in rows.values()])) for k in \'EBCD\'}\n        compare = [\'B\', \'C\'] + ([] if undec_D else [\'D\'])\n        beats = all(mean[\'E\'] < mean[k] for k in compare)\n        all_within = all(r[\'within\'] for r in rows.values())\n        v = \'判定不能\' if undec_range else (\'当たる\' if (beats and all_within) else \'外れる\')\n        out[\'E2\'] = dict(rows=rows, mean_err=mean, compared_with=compare, beats_baselines=beats,\n                         all_within=all_within, n_models=len(rows), verdict=v,\n                         excluded_no_crossing=[n for n in models if n not in have])\n    else:\n        out[\'E2\'] = dict(verdict=\'交差のあるモデルがない\', excluded_no_crossing=list(models))\n    # P2\n    names = list(models)\n    fams = [models[n][\'fam\'] for n in names]\n    Fs = [np.array(models[n][\'fb\'][\'F\']) for n in names]\n    x1 = [np.log(np.array(temps)) for n in names]\n    x3 = [np.array(models[n][\'ell_tf\']) for n in names]\n    sigma = float(np.median([models[n][\'fb\'][\'eps\'] / 1.96 for n in names]))\n    e1_, o1 = spline_error(x1, Fs, fams, sigma)\n    e3_, o3 = spline_error(x3, Fs, fams, sigma)\n    v = \'判定不能\' if undec_range else (\'成り立つ\' if e3_ < 0.7 * e1_ else \'成り立たない\')\n    out[\'P2\'] = dict(err_x1=e1_, err_x3=e3_, ratio=e3_ / e1_ if e1_ > 0 else float(\'nan\'), sigma=sigma,\n                     outside_x1=o1, outside_x3=o3, verdict=v)\n    out[\'Q1_crossing\'] = {n: bool(n in have) for n in models}\n    base_T0 = [T0s[n] for n in have if models[n][\'kind\'] == \'base\']\n    out[\'Q5_base_T0_median\'] = float(np.median(base_T0)) if base_T0 else float(\'nan\')\n    out[\'tag\'] = q_tag\n    return out\n\n\ndef analyze(d):\n    rng = np.random.default_rng(SEED)\n    res = {\'models\': {}}\n    raw = {}\n    for name, fam, kind in MODELS:\n        Z = np.load(f\'{d}/{name}.npz\')\n        temps = [float(t) for t in Z[\'temps\']]\n        tf, gen = Z[\'tf\'], Z[\'gen\']\n        Tcal, n_bad = t_cal(Z[\'tcal\'], Z[\'tcal_T\'])\n        a = float(np.median(Z[\'a\'])); h = float(np.median(Z[\'h\']))\n        m = dict(fam=fam, kind=kind, Tcal=Tcal, tcal_bad_docs=n_bad, a=a, h=h, n_docs=int(tf.shape[1]),\n                 H2=int(Z[\'H2\']), tok_len=int(Z[\'tok_len\']))\n        for q, key in [(0, \'H\'), (3, \'H2\')]:\n            fd = per_doc_feedback(tf[q], gen[q])\n            m[\'fb_\' + key] = feedback_stats(fd, temps, rng)\n            m[\'ell_tf_\' + key] = np.median(tf[q].astype(np.float64).reshape(-1, len(temps), tf.shape[-1])\n                                           .transpose(1, 0, 2).reshape(len(temps), -1), 1).tolist()\n        # 記録：1 位の log-odds とエントロピーの F、f(t) の曲線、distinct-4\n        for q, key in [(1, \'top1\'), (2, \'ent\')]:\n            fd = per_doc_feedback(tf[q], gen[q])\n            m[\'F_\' + key] = np.median(fd, 0).tolist()\n        g = gen[0].astype(np.float64).mean(1).transpose(1, 0, 2) - tf[0].astype(np.float64)   # (nD, nT, L)\n        bins = [(1, 8), (8, 32), (32, 128), (128, 512)]\n        m[\'f_by_position\'] = {f\'{a_}-{b_}\': np.median(g[:, :, a_:b_].mean(-1), 0).tolist() for a_, b_ in bins}\n        m[\'distinct4_last128\'] = distinct4(Z[\'toks\'])\n        m[\'ell_gen_mean\'] = gen[0].astype(np.float64)[:, :, :, 1:].mean((1, 2, 3)).tolist()   # (nT,)\n        res[\'models\'][name] = m\n        raw[name] = m\n    res[\'temps\'] = temps\n    for key in [\'H\', \'H2\']:\n        mm = {n: dict(fam=m[\'fam\'], kind=m[\'kind\'], fb=m[\'fb_\' + key], Tcal=m[\'Tcal\'], a=m[\'a\'], ell_tf=m[\'ell_tf_\' + key])\n              for n, m in raw.items()}\n        res[\'judge_\' + key] = judge(mm, temps, key)\n    # Q6：別のモデルの文の対照\n    A_all, B_all, rows = [], [], {}\n    for reader, writer in CROSS:\n        C = np.load(f\'{d}/cross_{reader}_reads_{writer}.npz\')[\'out\']           # (2, nT, S, nD)\n        cross = np.nanmean(C[0], (1, 2)); human = np.nanmean(C[1], (1, 2))\n        self_ = np.array(raw[reader][\'ell_gen_mean\'])\n        A = self_ - cross; B = cross - human\n        rows[f\'{reader}<-{writer}\'] = dict(self_minus_cross=A.tolist(), cross_minus_human=B.tolist())\n        A_all.append(np.abs(A)); B_all.append(np.abs(B))\n    A_m, B_m = float(np.mean(A_all)), float(np.mean(B_all))\n    if abs(A_m - B_m) <= 0.2 * max(A_m, B_m):\n        v = \'同じくらい\'\n    else:\n        v = \'自分の文のほうが大きい\' if A_m > B_m else \'変な文だからのほうが大きい\'\n    res[\'Q6\'] = dict(rows=rows, mean_abs_self=A_m, mean_abs_odd=B_m, verdict=v)\n    return res\n\n\ndef _clean(o):\n    if isinstance(o, dict):\n        return {k: _clean(v) for k, v in o.items()}\n    if isinstance(o, (list, tuple)):\n        return [_clean(v) for v in o]\n    if isinstance(o, (np.floating, float)):\n        return None if not np.isfinite(o) else float(f\'{float(o):.6g}\')\n    if isinstance(o, (np.integer,)):\n        return int(o)\n    if isinstance(o, np.bool_):\n        return bool(o)\n    return o\n\n\nif False:\n    r = analyze(sys.argv[1])\n    json.dump(_clean(r), open(sys.argv[2], \'w\'), ensure_ascii=False, separators=(\',\', \':\'))\n    print(\'wrote\', sys.argv[2])\n'
SRCA24 = '"""A2-4（ゲート v0.2、凍結 2026-10-28 18:28）の分析。scripts/a2_2_analyze.py を実行したのと同じ名前空間で実行する\n（pav・crossing_mono・feedback_stats・per_doc_feedback・distinct4・NBOOT を変えずに使う）。データを開く前に固定する。\n\n    analyze4(a2_4 のフォルダ, a2_2 のフォルダ) -> dict\n\n細部（データを開く前に決めた。ゲートに書いていないもの）：\n- f は per_doc_feedback（t ＝ 2〜512 の平均、記事ごと）。ε はモデルごとに 1 つ：\'none\' の全データで feedback_stats が出す ε。\n- 組にしたブートストラップは種 1、2,000 回。d_∞ は (block, none)、d_偽 は (placebo, none)、d_差 は (block, placebo) の組。\n  各回で両方が交差した回だけを使う。その割合が 0.9 未満、または使える回が 20 回未満なら「はっきりしない」。\n- d_差 の判定で、どちらかの T_0 が交差しない（格子の外）ときは「はっきりしない」。\n- 「10% 以上は動かない」：区間の下の端 ≥ −0.10、上の端 ≤ +0.10、区間が 0 を含む。\n- Q4：3 モデルのうち |d_偽| < 0.5 × |d_∞|（点）の数。どちらかの点が出ないモデルは数えない（記録する）。\n- Q5：温度 0.7、128〜512 トークン目の f（記事ごとに、生成 2 本の平均 − 人の文脈、その帯で平均）の記事の平均を、\n  3 モデルで足したもの同士の比（block ÷ none）。r < 0.25 →「ほぼ全部消える」、0.25〜0.75 →「半分くらい残る」、\n  0.75 より大きく 1 以下 →「ほとんど残る」、1 より大きい →「増える」。\n- 写しかけ歩（人の文）：続きの各位置で、直前の 3 トークンがそれより前（書き出しを含む）に出ていて、続いたトークンがあるか。\n- 結晶の分（写しかけ歩の印で）：記事ごとに、t ＝ 2〜512（または帯）の歩を、生成 2 本まとめて分ける。\n  結晶 ＝ c_gen ×（ℓ_gen（写しかけ）− ℓ_gen（ほか））− c_hum ×（ℓ_tf（写しかけ）− ℓ_tf（ほか））。片方の歩がないときはその項は 0。\n- d_機：\'none\' の記事ごとの f から結晶の分を引いて、同じ ε で T_0 を出し直す。|d_機| < 0.01 なら r は読まない。\n- 確かめ 1 が通らなければ、全体（QB1）だけでなく、モデルごとの判定（d_∞・d_差）も「はっきりしない」にする（ゲート §6 の 1 の「全体」をこう読む）。\n- 確かめ 4（偽薬の割合）は、温度ごとに |割合 − p(T)| ≤ 0.2 × p(T)。ただし禁止の数の差が 2 回以内なら通る（数が少ないときの揺れ）。\n  通らなくても判定は変えず、記録して結果ノートに書く（ゲートは確かめ 4 が外れたときの扱いを決めていない）。\n- 判定が「大きく下がる（格子の外）」「上がる（格子の外）」のときは、採点では「大きく下がる」「上がる」と同じに扱う。\n- Q5 の比が出ない（足した f が 0）ときは「読まない」。\n"""\nimport math\nimport numpy as np\n\nMODELS4 = [\'pythia-1b\', \'olmo2-1b\', \'qwen2.5-1.5b\']\nMODES4 = [\'none\', \'block\', \'placebo\']\nBAND10 = 0.10\n\n\ndef human_cand(prompt, human):\n    seq = [int(t) for t in prompt]\n    nxt, out = {}, np.zeros(len(human), bool)\n    for i in range(3, len(seq)):\n        nxt.setdefault(tuple(seq[i - 3:i]), set()).add(seq[i])\n    for t, y in enumerate(human):\n        if len(seq) >= 3:\n            out[t] = tuple(seq[-3:]) in nxt\n            nxt.setdefault(tuple(seq[-3:]), set()).add(int(y))\n        seq.append(int(y))\n    return out\n\n\ndef copy_frac(prompt, cont, n):\n    seq = [int(t) for t in prompt] + [int(t) for t in cont]\n    P, seen, c = len(prompt), set(), 0\n    for i in range(n - 1, len(seq)):\n        g = tuple(seq[i - n + 1:i + 1])\n        if i >= P and g in seen:\n            c += 1\n        seen.add(g)\n    return c / len(cont)\n\n\ndef stats_fixed(fd, temps, eps):\n    F = np.median(fd, 0)\n    g = pav(F)\n    T0 = crossing_mono(F, temps, eps)\n    cls = \'cross\' if np.isfinite(T0) else (\'below\' if g[0] >= eps else \'above\' if g[-1] <= -eps else \'flat\')\n    return dict(F=F.tolist(), F_pav=g.tolist(), T0=T0, cls=cls)\n\n\ndef paired(fdA, fdB, temps, eps, seed=1):\n    """log(T_0(A) ÷ T_0(B)) の点と、記事を組にしたブートストラップの 95% 区間。"""\n    a = crossing_mono(np.median(fdA, 0), temps, eps); b = crossing_mono(np.median(fdB, 0), temps, eps)\n    point = math.log(a / b) if np.isfinite(a) and np.isfinite(b) else float(\'nan\')\n    rng = np.random.default_rng(seed)\n    nD = fdA.shape[0]\n    lr = []\n    for _ in range(NBOOT):\n        idx = rng.integers(0, nD, nD)\n        x = crossing_mono(np.median(fdA[idx], 0), temps, eps); y = crossing_mono(np.median(fdB[idx], 0), temps, eps)\n        if np.isfinite(x) and np.isfinite(y):\n            lr.append(math.log(x / y))\n    frac = len(lr) / NBOOT\n    ci = [float(np.percentile(lr, 2.5)), float(np.percentile(lr, 97.5))] if len(lr) >= 20 else [float(\'nan\')] * 2\n    return dict(point=point, ci=ci, both_cross=frac)\n\n\ndef verdict_inf(p, cls_new, base_ok):\n    if not base_ok:\n        return \'はっきりしない\'\n    if cls_new == \'below\':\n        return \'大きく下がる（格子の外）\'\n    if cls_new == \'above\':\n        return \'上がる（格子の外）\'\n    if cls_new == \'flat\':\n        return \'はっきりしない\'\n    lo, hi = p[\'ci\']\n    if p[\'both_cross\'] < 0.9 or not np.isfinite(lo):\n        return \'はっきりしない\'\n    if hi < 0:\n        return \'大きく下がる\' if p[\'point\'] <= -BAND10 else \'少し下がる\'\n    if lo > 0:\n        return \'上がる\'\n    if lo >= -BAND10 and hi <= BAND10 and lo <= 0 <= hi:\n        return \'10% 以上は動かない\'\n    return \'はっきりしない\'\n\n\ndef verdict_diff(p, ok):\n    lo, hi = p[\'ci\']\n    if not ok or p[\'both_cross\'] < 0.9 or not np.isfinite(lo):\n        return \'はっきりしない\'\n    if hi < 0:\n        return \'写しを止めた分がある\'\n    if lo > 0:\n        return \'偽薬のほうが下げた\'\n    return \'偽薬と区別できない\'\n\n\ndef gen_mean(Z, key):\n    return Z[\'g_\' + key].astype(np.float64).mean(1)            # (nT, nD, L)、生成 2 本の平均\n\n\ndef crystal_part(Z, Hc, tf_ell, k, a, b):\n    """記事ごとの結晶の分（写しかけ歩の印で）。Z：生成の npz、Hc：(nD, L) 人の写しかけ、tf_ell：(nD, nT, L)。→ (nD,)"""\n    ge = Z[\'g_ell\'][k][:, :, a:b].astype(np.float64)            # (S, nD, L\')\n    gc = Z[\'g_cand\'][k][:, :, a:b]\n    nD = ge.shape[1]\n    out = np.zeros(nD)\n    for j in range(nD):\n        x, m = ge[:, j].ravel(), gc[:, j].ravel()\n        y, h = tf_ell[j, k, a:b].astype(np.float64), Hc[j, a:b]\n        cg, ch = m.mean(), h.mean()\n        part_g = cg * (x[m].mean() - x[~m].mean()) if (m.any() and (~m).any()) else 0.0\n        part_h = ch * (y[h].mean() - y[~h].mean()) if (h.any() and (~h).any()) else 0.0\n        out[j] = part_g - part_h\n    return out\n\n\ndef analyze4(d4, d2):\n    temps = None\n    res = {\'models\': {}, \'checks\': {}}\n    f07 = {m: {} for m in MODES4}\n    for name in MODELS4:\n        Z = {m: (lambda z: {k: z[k] for k in z.files})(np.load(f\'{d4}/{name}@{m}.npz\')) for m in MODES4}\n        Z2 = (lambda z: {k: z[k] for k in z.files})(np.load(f\'{d2}/{name}.npz\'))\n        temps = [float(t) for t in Z[\'none\'][\'temps\']]\n        k07 = temps.index(0.7)\n        tf_ell = Z[\'none\'][\'tf\'][0]\n        # 確かめ 1：A2-2 と同じ測り方\n        assert np.array_equal(Z[\'none\'][\'use\'], Z2[\'use\']) and Z[\'none\'][\'tf\'].shape == Z2[\'tf\'].shape, \'記事の組が A2-2 と違う\'\n        c1 = {\'tf_max_abs_diff\': float(np.max(np.abs(Z[\'none\'][\'tf\'][0].astype(np.float64) - Z2[\'tf\'][0].astype(np.float64))))}\n        s_new = feedback_stats(per_doc_feedback(tf_ell, Z[\'none\'][\'g_ell\']), temps, np.random.default_rng(0))\n        s_old = feedback_stats(per_doc_feedback(Z2[\'tf\'][0], Z2[\'gen\'][0]), temps, np.random.default_rng(0))\n        c1[\'T0_new_own_eps\'], c1[\'T0_a22\'] = s_new[\'T0\'], s_old[\'T0\']\n        c1[\'log_ratio\'] = math.log(s_new[\'T0\'] / s_old[\'T0\']) if np.isfinite(s_new[\'T0\']) and np.isfinite(s_old[\'T0\']) else float(\'nan\')\n        c1[\'ok\'] = bool(c1[\'tf_max_abs_diff\'] <= 1e-3 and np.isfinite(c1[\'log_ratio\']) and abs(c1[\'log_ratio\']) <= 0.05)\n        # 確かめ 3：block で写し歩が 0\n        P, Hm = Z[\'none\'][\'prompts\'], Z[\'none\'][\'humans\']\n        nT, S, nD, L = Z[\'block\'][\'toks\'].shape\n        n_copy = sum(copy_frac(P[j], Z[\'block\'][\'toks\'][k, s, j], 4) > 0 for k in range(nT) for s in range(S) for j in range(nD))\n        # 確かめ 4：偽薬の割合\n        pp = Z[\'placebo\'][\'p_placebo\']\n        rate = Z[\'placebo\'][\'g_banned_top1\'].reshape(nT, -1).mean(1)\n        Nst = Z[\'placebo\'][\'g_banned_top1\'][0].size\n        rel = [0.0 if abs(r - p) * Nst <= 2 else (abs(r / p - 1) if p > 0 else float(\'inf\')) for r, p in zip(rate, pp)]\n        # 確かめ 5：1 歩目のつまみ前の ℓ が同じ\n        e0 = [Z[m][\'g_ell\'][:, :, :, 0] for m in MODES4]\n        c5 = bool(np.allclose(e0[0], e0[1], atol=1e-4) and np.allclose(e0[0], e0[2], atol=1e-4))\n        # 確かめ 2（データの側）：block で禁止した歩（禁止した確率 > 1e-3）では、つまみ後の ℓ がつまみ前と違う\n        mb = Z[\'block\'][\'g_banned\'] & (Z[\'block\'][\'g_ban_mass\'] > 1e-3)\n        c2 = float(np.mean(np.abs(Z[\'block\'][\'g_ell_pen\'][mb] - Z[\'block\'][\'g_ell\'][mb]) > 1e-6)) if mb.any() else float(\'nan\')\n        res[\'checks\'][name] = dict(check1=c1, check3_block_seqs_with_copy=int(n_copy), check4_rate=rate.tolist(), check4_p=pp.tolist(),\n                                   check4_ok=bool(all(x <= 0.2 for x in rel)), check5_first_step_same=c5,\n                                   check2_frac_pen_differs=c2, check2_ok=bool(np.isnan(c2) or c2 >= 0.99))\n        # 判定\n        fd = {m: per_doc_feedback(tf_ell, Z[m][\'g_ell\']) for m in MODES4}\n        eps = s_new[\'eps\']\n        st = {m: stats_fixed(fd[m], temps, eps) for m in MODES4}\n        base_ok = st[\'none\'][\'cls\'] == \'cross\' and c1[\'ok\']          # 確かめ 1 が通らなければ、モデルごとの判定も「はっきりしない」\n        p_inf = paired(fd[\'block\'], fd[\'none\'], temps, eps)\n        p_pl = paired(fd[\'placebo\'], fd[\'none\'], temps, eps)\n        p_df = paired(fd[\'block\'], fd[\'placebo\'], temps, eps)\n        m = dict(eps=eps, stats=st, d_inf=p_inf, d_placebo=p_pl, d_diff=p_df,\n                 verdict_inf=verdict_inf(p_inf, st[\'block\'][\'cls\'], base_ok),\n                 verdict_diff=verdict_diff(p_df, base_ok and st[\'block\'][\'cls\'] == \'cross\' and st[\'placebo\'][\'cls\'] == \'cross\'))\n        # d_機\n        Hc = np.stack([human_cand(P[j], Hm[j]) for j in range(nD)])\n        cr = np.stack([crystal_part(Z[\'none\'], Hc, tf_ell, k, 1, L) for k in range(nT)], 1)      # (nD, nT)\n        T0m = crossing_mono(np.median(fd[\'none\'] - cr, 0), temps, eps)\n        m[\'T0_mech\'] = T0m\n        m[\'d_mech\'] = math.log(T0m / st[\'none\'][\'T0\']) if np.isfinite(T0m) and base_ok else float(\'nan\')\n        dm, dd = m[\'d_mech\'], p_df[\'point\']\n        if np.isfinite(dm) and abs(dm) >= 0.01 and np.isfinite(dd):\n            r = dd / dm\n            m[\'r\'] = r\n            m[\'r_reading\'] = (\'フィードバックで増えた\' if r > 1.5 else \'ほぼ見積もりどおり\' if r >= 0.67 else\n                              \'打ち消された\' if r >= 0 else \'逆向き\')\n        else:\n            m[\'r\'], m[\'r_reading\'] = float(\'nan\'), \'読まない\'\n        # 記録\n        for mo in MODES4:\n            g = gen_mean(Z[mo], \'ell\') - tf_ell.transpose(1, 0, 2)                                  # (nT, nD, L)\n            m.setdefault(\'f_by_position\', {})[mo] = {f\'{a}-{b}\': np.median(g[:, :, a:b].mean(-1), 1).tolist()\n                                                     for a, b in [(1, 8), (8, 32), (32, 128), (128, 512)]}\n            f07[mo][name] = float(g[k07, :, 128:512].mean(-1).mean())\n            for q, key in [(1, \'top1\'), (2, \'ent\')]:\n                m.setdefault(\'F_\' + key, {})[mo] = np.median(per_doc_feedback(Z[\'none\'][\'tf\'][q], Z[mo][\'g_\' + key]), 0).tolist()\n            m.setdefault(\'distinct4\', {})[mo] = distinct4(Z[mo][\'toks\'])\n            m.setdefault(\'cand_rate\', {})[mo] = Z[mo][\'g_cand\'].reshape(nT, -1).mean(1).tolist()\n            m.setdefault(\'banned_rate\', {})[mo] = Z[mo][\'g_banned\'].reshape(nT, -1).mean(1).tolist()\n            m.setdefault(\'ban_mass_mean\', {})[mo] = Z[mo][\'g_ban_mass\'].reshape(nT, -1).mean(1).tolist()\n            m.setdefault(\'crystal_part_07_08_late\', {})[mo] = {\n                str(T): float(crystal_part(Z[mo], Hc, tf_ell, temps.index(T), 128, 512).mean()) for T in [0.7, 0.8]}\n            m.setdefault(\'copy_frac_n\', {})[mo] = {\n                f\'n{n}\': {str(T): float(np.mean([copy_frac(P[j], Z[mo][\'toks\'][temps.index(T), s, j], n)\n                                                for s in range(S) for j in range(nD)])) for T in [0.7, 0.8]} for n in [3, 4, 8]}\n            m.setdefault(\'stats_raw_pen_T0\', {})[mo] = crossing_mono(np.median(per_doc_feedback(tf_ell, Z[mo][\'g_ell_pen\']), 0), temps, eps)\n        m[\'copy_frac_n_human\'] = {f\'n{n}\': float(np.mean([copy_frac(P[j], Hm[j], n) for j in range(nD)])) for n in [3, 4, 8]}\n        res[\'models\'][name] = m\n    res[\'temps\'] = temps\n    # 全体\n    ok_all = all(c[\'check1\'][\'ok\'] for c in res[\'checks\'].values())\n    A = [n for n in MODELS4 if res[\'models\'][n][\'verdict_diff\'] == \'写しを止めた分がある\']\n    Bm = [n for n in MODELS4 if res[\'models\'][n][\'verdict_diff\'] == \'偽薬と区別できない\' and res[\'models\'][n][\'verdict_inf\'] == \'10% 以上は動かない\']\n    if not ok_all:\n        qb1 = \'はっきりしない（確かめ 1 が通らなかった）\'\n    elif len(A) == 3:\n        qb1 = \'4 トークンの完全な写しは T_0 を支えている\'\n    elif len(Bm) == 3:\n        qb1 = \'T_0 は 4 トークンの完全な写しでは決まっていない\'\n    elif A and Bm:\n        qb1 = \'モデルによる\'\n    else:\n        qb1 = \'はっきりしない\'\n    res[\'QB1\'] = qb1\n    res[\'check1_all_ok\'] = ok_all\n    cnt, skipped = 0, []\n    for n in MODELS4:\n        a, b = res[\'models\'][n][\'d_placebo\'][\'point\'], res[\'models\'][n][\'d_inf\'][\'point\']\n        if np.isfinite(a) and np.isfinite(b):\n            cnt += int(abs(a) < 0.5 * abs(b))\n        else:\n            skipped.append(n)\n    res[\'Q4\'] = dict(count=cnt, skipped=skipped)\n    sn, sb = sum(f07[\'none\'].values()), sum(f07[\'block\'].values())\n    r5 = sb / sn if sn != 0 else float(\'nan\')\n    res[\'Q5\'] = dict(ratio=r5, f07=f07, answer=(\'読まない（比が出ない）\' if not np.isfinite(r5) else \'増える\' if r5 > 1 else \'ほとんど残る\' if r5 > 0.75 else\n                                               \'半分くらい残る\' if r5 >= 0.25 else \'ほぼ全部消える\'))\n    dm = {n: res[\'models\'][n][\'d_mech\'] for n in MODELS4}\n    dd = {n: res[\'models\'][n][\'d_diff\'][\'point\'] for n in MODELS4}\n    res[\'order_record\'] = dict(d_mech=dm, d_diff=dd)\n    res[\'Q1_3\'] = {n: res[\'models\'][n][\'verdict_inf\'] for n in MODELS4}\n    return res\n\n\ndef retok_record(d4, hf_ids, T=0.7):\n    """記録（ゲート §7）：生成した続きをデコードしてエンコードし直したとき、トークン列が変わる文の割合（温度 T）。"""\n    from transformers import AutoTokenizer\n    out = {}\n    for name in MODELS4:\n        tok = AutoTokenizer.from_pretrained(hf_ids[name])\n        out[name] = {}\n        for mo in MODES4:\n            Z = np.load(f\'{d4}/{name}@{mo}.npz\')\n            k = [float(t) for t in Z[\'temps\']].index(T)\n            toks = Z[\'toks\'][k]\n            diff = [list(tok(tok.decode(toks[s, j]), add_special_tokens=False).input_ids) != [int(t) for t in toks[s, j]]\n                    for s in range(toks.shape[0]) for j in range(toks.shape[1])]\n            out[name][mo] = float(np.mean(diff))\n    return out\n'
assert hashlib.sha256(SRCA22.encode()).hexdigest() == 'fe1f96db1f7e84031e85a70bcc8dcbb95c9fc06b3367657ff4074a9f9d1343f7'
assert hashlib.sha256(SRCA24.encode()).hexdigest() == 'e70bad0db9b7780c2059aae52ad06debe87757bc582791deefb58c5f435f84a6'
exec(SRCA22)
exec(SRCA24)
print('コードを置きました')


In [ ]:
# セル3：分析して保存（結果は表示しない）
import json
r = analyze4('/content/drive/MyDrive/logit-thermo/A2_4', '/content/drive/MyDrive/logit-thermo' + '/A2_2')
r['retok'] = retok_record('/content/drive/MyDrive/logit-thermo/A2_4', {'pythia-1b': 'EleutherAI/pythia-1b', 'olmo2-1b': 'allenai/OLMo-2-0425-1B', 'qwen2.5-1.5b': 'Qwen/Qwen2.5-1.5B'})
r['a2_4_analyze_sha256'] = 'e70bad0db9b7780c2059aae52ad06debe87757bc582791deefb58c5f435f84a6'
r['a2_2_analyze_sha256'] = '35142703f6b88d7a001bfe5ff3608ea9c6cb2e60c457fc02beb235fba946f20e'
json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo/A2_4' + '/a2_4_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
print('保存しました')


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：判定と要約を a2_4_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
